<a href="https://colab.research.google.com/github/hjiwoong/DL/blob/main/day06_practice1_%EC%A1%B0%EA%B8%B0%EC%A2%85%EB%A3%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 조기종료(Early Stopping) - 과적합이 시작되기 전에 멈춘다

import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# 셀 1. 데이터 - 피마 당뇨 데이터 로드 + 60/20/20 분할
CSV_URL  = "https://raw.githubusercontent.com/taehojo/deeplearning_4th/master/data/pima-indians-diabetes3.csv"
CSV_PATH = "pima-indians-diabetes3.csv"

def load_pima():
  import os
  if os.path.exists(CSV_PATH):
    print(f"로컬 파일 재사용: {CSV_PATH}")
    return pd.read_csv(CSV_PATH)

  try:
    df = pd.read_csv(CSV_URL)
    df.to_csv(CSV_PATH, index = False)
    print(f"다운로드 완료 → {CSV_PATH} 저장 (다음 실행부턴 재사용)")
    return df
  except Exception as e:
    raise SystemExit(
        f"데이터 다운로드 실패: {e}\n"
    )

df = load_pima()
X, y = df.iloc[:, :-1].values, df.iloc[:, -1].values

다운로드 완료 → pima-indians-diabetes3.csv 저장 (다음 실행부턴 재사용)


In [ ]:
# 데이터를 학습 60% / 검증 20% / 테스트 20% 로 나눈다
X_tr, X_tmp, y_tr, y_tmp = train_test_split(
    X, y, test_size=0.4,
    random_state=42, stratify=y # 클래스 비율 유지
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.5,
    random_state=42, stratify=y_tmp
)

scaler = StandardScaler()
def to_t(Xa, ya, fit=False):
  Xs = scaler.fit_transform(Xa) if fit else scaler.transform(Xa)
  return (torch.tensor(Xs, dtype=torch.float32).to(device),
          torch.tensor(ya, dtype=torch.float32).reshape(-1, 1).to(device))
X_tr_t, y_tr_t = to_t(X_tr, y_tr, fit=True)
X_val_t, y_val_t = to_t(X_val, y_val)
X_te_t, y_te_t = to_t(X_te, y_te)

print(f"학습 {len(X_tr)} / 검증 {len(X_val)} / 테스트 {len(X_te)}")

학습 460 / 검증 154 / 테스트 154


In [ ]:
# 셀 2. 모델,손실,정확도 함수 저의 - 과적합 유도 모델
def make_model():
  return nn.Sequential(
      nn.Linear(8, 128),nn.ReLU(),
      nn.Linear(128, 64),nn.ReLU(),
      nn.Linear(64, 1), nn.Sigmoid())

loss_fn = nn.BCELoss()
def accuracy(model, Xt, yt):
  model.eval()
  with torch.no_grad():
    return ((model(Xt) > 0.5) == yt.bool()).float().mean().item()

In [ ]:
# 셀 3. 500 애폭 완주 - 기준선
torch.manual_seed(42)
model = make_model().to(device)
opt = torch.optim.Adam(model.parameters(), lr=0.005)

for epoch in range(500):
  model.train()
  loss = loss_fn(model(X_tr_t), y_tr_t)
  opt.zero_grad(); loss.backward(); opt.step()

model.eval()
with torch.no_grad():
  last_val = loss_fn(model(X_val_t), y_val_t).item()
print(f"[완주 500ep] val loss {last_val:.3f} | 테스트 정확도 {accuracy(model, X_te_t, y_te_t):.3f}")

[완주 500ep] val loss 4.717 | 테스트 정확도 0.669


In [ ]:
# 셀 4. 조기종료(Early Stopping) - patience 로 자동 중단

def train_early_stop(max_epochs=500, patience=30):
  torch.manual_seed(42)
  model = make_model().to(device)
  opt = torch.optim.Adam(model.parameters(), lr=0.005)

  best_val, best_state, wait = float("inf"), None, 0

  for epoch in range(max_epochs):
    model.train()
    loss = loss_fn(model(X_tr_t), y_tr_t)
    opt.zero_grad(); loss.backward(); opt.step()

    model.eval()
    with torch.no_grad():
      val_loss = loss_fn(model(X_val_t), y_val_t).item()

    if val_loss < best_val:
      best_val = val_loss
      best_state = {k: v.clone() for k, v in model.state_dict().items()} #가중치 '복사' 저장 - {"key": value}, 최고 모델의 가중치(weight)가 딕셔너리
      wait = 0
    else:   # best_val을 갱신을 못하면
      wait += 1
      if wait >= patience:
        print(f" EarlyStopping: {epoch+1} 애폭에서 중단 " f"(best 는 {epoch+1-patience} 애폭 근처)")
        break

  model.load_state_dict(best_state) #best 시점 가중치로 복원
  return model, best_val

model_es, best_val = train_early_stop()
print(f"[조기종료] val loss {best_val:.3f} | 테스트 정확도 {accuracy(model_es, X_te_t, y_te_t):.3f}")

 EarlyStopping: 55 애폭에서 중단 (best 는 25 애폭 근처)
[조기종료] val loss 0.428 | 테스트 정확도 0.721
